# Transcriptômica Espacial

## Links para Tutoriais
Explore os seguintes tutoriais para começar a trabalhar com análise transcriptômica espacial:

* Pacote de Ferramentas de Transcriptômica Espacial [semla](https://ludvigla.github.io/semla/articles/getting_started.html). Além deste tutorial, existem outros dos quais grande parte deste notebook foi derivada, abordando extração e análise transcriptômica espacial com o pacote semla.

* [Vignette de Transcriptômica Espacial do Seurat](https://satijalab.org/seurat/articles/spatial_vignette.html) Como realizar análise transcriptômica espacial com o pacote Seurat?

# Instale os pacotes

In [ ]:
## Função para executar comandos de terminal no Google Colab com o kernel de R
shell_call <- function(command, ...) {
  result <- system(command, intern = TRUE, ...)  # Executa o comando de terminal e guarda a saída
  cat(paste0(result, collapse = "\n"))  # Imprime a saída em um formato legível
}
# Baixa o código da URL especificada e o salva como "add_cranapt_jammy.sh"
download.file("https://github.com/eddelbuettel/r2u/raw/master/inst/scripts/add_cranapt_jammy.sh",
              "add_cranapt_jammy.sh")
Sys.chmod("add_cranapt_jammy.sh", "0755")
shell_call("./add_cranapt_jammy.sh")
bspm::enable()
options(bspm.version.check=FALSE)
shell_call("rm add_cranapt_jammy.sh")
# Define um limite de tempo mais alto para evitar interrupções durante downloads de pacotes.
options(timeout=1000)


In [ ]:
# Instala o pacote "semla" do GitHub forçando sua atualização
remotes::install_github("ludvigla/semla", upgrade=T, force = TRUE)

In [ ]:
# Instala pacotes
cranPkgs2Install = c("BiocManager", "ggpubr", "Seurat", "hdf5r", "openxlsx",
                     "enrichR", "clustree", "DT","ggcorrplot","scatterpie", "pheatmap")
install.packages(cranPkgs2Install, ask=FALSE, update=TRUE, quietly=TRUE)

In [ ]:
biocPkgs2Install = c("SingleCellExperiment", "ReactomePA", "org.Hs.eg.db", "glmGamPoi", "fgsea", "limma")
BiocManager::install(biocPkgs2Install, ask=FALSE, update=TRUE, quietly=TRUE)

In [ ]:
# Carrega pacotes
# Suprime as mensagens ao carregar pacotes para manter a saída limpa.
suppressPackageStartupMessages({
library(Seurat)  # Framework de análise para RNA-seq de célula única.
library(data.table)  # Manipulação eficiente de grandes volumes de dados.
library(ggplot2)  # Visualização baseada na gramática de gráficos.
library(plotly)  # Visualizações interativas.
library(RColorBrewer)  # Paletas de cores predefinidas.
library(dplyr)  # Manipulação de dados.
library(semla) # Ferramentas para transcriptômica espacial.
library(clustree)  # Visualização de resoluções de clusters.
library(ReactomePA)  # Análise de vias metabólicas.
library(org.Hs.eg.db)  # Banco de dados de anotação gênica humana.
library(ggpubr)  # Visualizações prontas para publicação.
library(enrichR)  # Análise de enriquecimento gênico.
library(stringr)  # Manipulação de cadeias de caracteres.
library(openxlsx)  # Manipulação de arquivos Excel.
library(patchwork)  # Combina múltiplos gráficos.
library(SingleCellExperiment)  # Estrutura para dados de célula única.
})


# Introdução
A transcriptômica espacial é uma técnica que combina informações sobre a expressão gênica com a localização física das células dentro de um tecido. Ao contrário dos métodos transcriptômicos tradicionais, aqui sabemos não apenas quais genes são expressos, mas também onde são expressos, permitindo um estudo mais preciso da organização celular e dos microambientes teciduais.

Nesse contexto, as imagens histológicas desempenham um papel fundamental. Uma das mais utilizadas é a coloração H&E (Hematoxilina e Eosina), que colore os tecidos para destacar sua estrutura:

* A hematoxilina cora os núcleos celulares de azul/roxo.

* A eosina cora o citoplasma e os componentes extracelulares de rosa.

A combinação da transcriptômica espacial com imagens H&E permite relacionar perfis moleculares com a arquitetura tecidual, oferecendo uma visão abrangente de como as células estão organizadas e funcionam em seu ambiente natural.

## Baixe os dados

Os dados baixados abaixo correspondem a um conjunto de dados de transcriptômica espacial disponível no site da 10x e gerado com o Space Ranger (https://www.10xgenomics.com/support/software/space-ranger/latest). A amostra é de tecido mamário canceroso.

Você pode consultar os detalhes nos seguintes links:

- [Câncer de Mama Humano (Bloco A, Seção 1)](https://www.10xgenomics.com/datasets/human-breast-cancer-block-a-section-1-1-standard-1-1-0)

- [Câncer de Mama Humano (Bloco A, Seção 2)](https://www.10xgenomics.com/datasets/human-breast-cancer-block-a-section-2-1-standard-1-1-0)

In [ ]:
shell_call("mkdir -p ST_Exercises")

# Baixe os arquivos que contém exercícios de transcriptômica espacial
download.file('https://cf.10xgenomics.com/samples/spatial-exp/1.1.0/V1_Breast_Cancer_Block_A_Section_2/V1_Breast_Cancer_Block_A_Section_2_filtered_feature_bc_matrix.h5','ST_Exercises/Breast_Cancer_Block_A_Section_2_filtered_feature_bc_matrix.h5')
download.file('https://cf.10xgenomics.com/samples/spatial-exp/1.1.0/V1_Breast_Cancer_Block_A_Section_2/V1_Breast_Cancer_Block_A_Section_2_spatial.tar.gz','ST_Exercises/Breast_Cancer_Block_A_Section_2_spatial.tar.gz')
download.file('https://cf.10xgenomics.com/samples/spatial-exp/1.1.0/V1_Breast_Cancer_Block_A_Section_2/V1_Breast_Cancer_Block_A_Section_2_metrics_summary.csv','ST_Exercises/Breast_Cancer_Block_A_Section_2_metrics_summary.csv')
download.file('https://cf.10xgenomics.com/samples/spatial-exp/1.1.0/V1_Breast_Cancer_Block_A_Section_2/V1_Breast_Cancer_Block_A_Section_2_web_summary.html','ST_Exercises/Breast_Cancer_Block_A_Section_2_web_summary.html')

download.file('https://cf.10xgenomics.com/samples/spatial-exp/1.1.0/V1_Breast_Cancer_Block_A_Section_1/V1_Breast_Cancer_Block_A_Section_1_filtered_feature_bc_matrix.h5','ST_Exercises/Breast_Cancer_Block_A_Section_1_filtered_feature_bc_matrix.h5')
download.file('https://cf.10xgenomics.com/samples/spatial-exp/1.1.0/V1_Breast_Cancer_Block_A_Section_1/V1_Breast_Cancer_Block_A_Section_1_spatial.tar.gz','ST_Exercises/Breast_Cancer_Block_A_Section_1_spatial.tar.gz')
download.file('https://cf.10xgenomics.com/samples/spatial-exp/1.1.0/V1_Breast_Cancer_Block_A_Section_1/V1_Breast_Cancer_Block_A_Section_1_metrics_summary.csv','ST_Exercises/Breast_Cancer_Block_A_Section_1_metrics_summary.csv')
download.file('https://cf.10xgenomics.com/samples/spatial-exp/1.1.0/V1_Breast_Cancer_Block_A_Section_1/V1_Breast_Cancer_Block_A_Section_1_web_summary.html','ST_Exercises/Breast_Cancer_Block_A_Section_1_web_summary.html')

# Lista os aquivos no diretório atual com o tamanho 
shell_call("ls -lh ST_Exercises")

In [ ]:
shell_call("tar -xvzf ST_Exercises/Breast_Cancer_Block_A_Section_1_spatial.tar.gz -C ST_Exercises/")
shell_call("mv ST_Exercises/spatial ST_Exercises/Spatial_Section_1")
shell_call("tar -xvzf ST_Exercises/Breast_Cancer_Block_A_Section_2_spatial.tar.gz -C ST_Exercises/")
shell_call("mv ST_Exercises/spatial ST_Exercises/Spatial_Section_2")

## Carregar o conjunto de dados

In [ ]:
samples <- list.files(
  path = "/content/ST_Exercises",
  pattern = "filtered_feature_bc_matrix.h5",
  full.names = TRUE,
  recursive = TRUE
)

imgs <- list.files(
  path = "/content/ST_Exercises",
  pattern = "tissue_lowres_image.png",
  full.names = TRUE,
  recursive = TRUE
)

spotfiles <- list.files(
  path = "/content/ST_Exercises",
  pattern = "positions_list.csv",
  full.names = TRUE,
  recursive = TRUE
)

json <- list.files(
  path = "/content/ST_Exercises",
  pattern = "scalefactors_json.json",
  full.names = TRUE,
  recursive = TRUE
)

infoTable <- tibble(samples, imgs, spotfiles, json, # Add required columns
                    sample_id = c("Breast_Cancer_Block_A_Section_1", "Breast_Cancer_Block_A_Section_2")) # Add additional column
dim(infoTable)

In [ ]:
# Carregar matrizes
SpatialData <- ReadVisiumData(infoTable)
SpatialData

In [ ]:
# Carrega um objeto do Seurat.
# load(file = "ST_Exercises/Exercise1_dataset.RData")
# Obtém informações do conjunto de dados

# Este comando recupera o ensaio padrão atual do objeto Seurat SpatialData
DefaultAssay(SpatialData)

# Mostra o número de genes (features) e células.
dim(SpatialData)
head(SpatialData@meta.data)

# Este comando gera uma tabela de frequências da coluna "ids" 
# nos metadados do objeto Seurat SpatialData.
table(SpatialData@meta.data$sample_id)


## Gráficos de Controle de Qualidade (Quality Control - QC)

Gráfico das métricas

In [ ]:
# Cria uma função de paleta de cores usando colorRampPalette do pacote grDevices, e cores do esquema "Set1" de RColorBrewer.
getPalette = colorRampPalette(brewer.pal(8, "Set1"))
color = getPalette(6)

VlnPlot(SpatialData, # Gera um violin plot (gráfico de violino)
       features = "nCount_Spatial", # Especifica a característica a ser plotada
       group.by = "sample_id", # Agrupa os dados de acordo com a coluna "ids" dos metadados
       pt.size = 0.1, # Define o tamanho dos pontos no gráfico
       cols = color) + # Define as cores para cada grupo
  stat_summary(fun.y=mean, geom="point", shape=95, size=15, color = "black") 
  + NoLegend()
  # Adiciona uma camada ao gráfico com a média da característica representada


In [ ]:
# Gráfico de dispersão que compara a quantidade de RNA com o número de características (genes)
FeatureScatter(object = SpatialData, feature1 = "nCount_Spatial",
               feature2 = "nFeature_Spatial", group.by = "sample_id", cols = color)


In [ ]:
# Carregar imagens H&E
SpatialData <- LoadImages(SpatialData)
ImagePlot(SpatialData)

In [ ]:
# Mapear as células do tecido
MapFeatures(SpatialData, features = "nFeature_Spatial",
            image_use = "raw", override_plot_dims = TRUE) & ThemeLegendRight()

In [ ]:
MapFeaturesSummary(
  object = SpatialData,          # seu objeto semla/Seurat
  features = "nCount_Spatial",   # métrica a ser mostrada
  pt_size = 2.5,                 # tamanho dos pontos
  ncol = 2,                      # número de colunas no layout
  subplot_type = "histogram"     # tipo de gráfico
)


### A alternativa do **Seurat** para representar graficamente dados de transcriptômica espacial

In [ ]:
# criar uma cópia para trabalhar
SpatialData2 <- SpatialData

# Carregar arquivos em um slot
SpatialData2@images[["slice1"]] <- Read10X_Image(dirname(SpatialData2@tools$Staffli@imgs[1]))
SpatialData2@images[["slice2"]] <- Read10X_Image(dirname(SpatialData2@tools$Staffli@imgs[2]))

# Visualizar os dados
SpatialFeaturePlot(SpatialData2, features = "nCount_Spatial") + 
                   theme(legend.position = "right")

# remover a cópia
rm(SpatialData2)


A normalização é uma etapa crucial de pré-processamento na análise de dados scRNA-seq. Ela envolve o ajuste dos dados brutos de expressão gênica para compensar variações técnicas e diferenças na profundidade de sequenciamento entre as células. Essas comparações devem ajudar a compreender os pontos fortes e as limitações de cada método de normalização.

Método de Normalização

* SCTransform: Um método mais avançado que utiliza regressão binomial negativa para remover a variabilidade técnica e estabilizar a variância, proporcionando uma normalização robusta para dados de célula única.

* Normalização Logarítmica: Escala os valores de expressão gênica e aplica uma transformação logarítmica, o que ajuda a estabilizar a variância e torna os dados mais adequados para análises posteriores.

Remoção da Variabilidade Técnica

* SCTransform: Maior robustez na remoção da variabilidade técnica, tornando-o ideal para conjuntos de dados com variação significativa.

* NormalizeData: Eficaz, porém menos sofisticado que o SCTransform no tratamento da variabilidade técnica.

Complexidade e Tempo de Execução

* SCTransform: Mais complexo e trabalhoso devido aos cálculos de regressão e à transformação para estabilização da variância.

* NormalizeData: Mais simples e rápido, ideal para análises rápidas e grandes conjuntos de dados.

Manipulação de Covariáveis

* SCTransform: Permite a inclusão de covariáveis ​​(por exemplo, número de genes detectados por célula) para regressão, o que melhora a qualidade dos dados normalizados.

* NormalizeData: Normalmente não incorpora covariáveis ​​diretamente no processo de normalização.

Estabilização da Variância

* SCTransform: Estabiliza a variância, tornando os dados mais adequados para análises subsequentes, como agrupamento e identificação de marcadores.

* NormalizeData: Usa transformação logarítmica para estabilizar parcialmente a variância, mas não tão eficazmente quanto o SCTransform.

Flexibilidade

* SCTransform: Concentra-se principalmente na manipulação de dados de sequenciamento de RNA de célula única com métodos de normalização robustos.

* NormalizeData: Flexível com diferentes métodos de normalização (por exemplo, LogNormalize), permitindo ajustes com base em necessidades específicas de análise.

Escalabilidade

* SCTransform: Pode lidar com grandes conjuntos de dados, mas pode ser mais lento devido à sua complexidade.

* NormalizeData: Lida com grandes conjuntos de dados de forma eficiente graças à sua simplicidade e velocidade.

In [ ]:
# Normalizar dados usando LogNormalize com um fator de escala
SpatialData <- NormalizeData(object = SpatialData, assay = "Spatial")

In [ ]:
SpatialData <- FindVariableFeatures(SpatialData, nfeatures = 10000)

> A comparação a seguir leva bastante tempo, **recomenda-se não ter executar.**

In [ ]:
# Normalizar dados usando SCTransform
SpatialData <- SCTransform(SpatialData, assay = "Spatial", verbose = TRUE, return.only.var.genes = FALSE)

# Comparar métodos de normalização (LogNormalize vs SCTransform)
SpatialData <- GroupCorrelation(SpatialData, group.assay = "Spatial", assay = "Spatial", layer = "data", do.plot = FALSE)
SpatialData <- GroupCorrelation(SpatialData, group.assay = "Spatial", assay = "SCT", layer = "scale.data", do.plot = FALSE)

# Gerar gráficos de comparação
p1 <- GroupCorrelationPlot(SpatialData, assay = "Spatial", cor = "nCount_Spatial_cor") + ggtitle("Log Normalization")
p2 <- GroupCorrelationPlot(SpatialData, assay = "SCT", cor = "nCount_Spatial_cor") + ggtitle("SCTransform Normalization")

# Mostrar ambos gráficos lado a lado
p1 + p2


## Agrupamento (Clustering)


In [ ]:
# Instalar pacotes por GitHub
devtools::install_github("zdebruine/RcppML")
devtools::install_github("zdebruine/singlet")

In [ ]:
# carregar pacote
library(singlet)

# Definir a semente para reprodutibilidade
set.seed(42)
SpatialData2 <- SpatialData

# OPCIONAL: criar subconjunto de dados para melhorar a velocidade de cálculo
SpatialData2 <- SpatialData2[VariableFeatures(SpatialData2), ]
SpatialData2 <- RunNMF(SpatialData2)


In [ ]:
# transfere as reduções
SpatialData@reductions  <- SpatialData2@reductions

# remove a cópia
rm(SpatialData2)
gc()

# transfere as colunas
k <- ncol(SpatialData@reductions$nmf@feature.loadings)
k


`RankPlot(SpatialData)` cria uma figura que mostra a classificação dos fatores NMF de acordo com sua contribuição, ajudando a selecionar os mais relevantes para interpretar padrões espaciais de expressão.

In [ ]:
# RankPlot cria uma figura com um gráfico de dados de classificação
RankPlot(SpatialData)

Veremos como os padrões de expressão estão distribuídos espacialmente no tecido, o que facilita a interpretação da heterogeneidade celular.

In [ ]:
# Ajusta as dimensões dos gráficos que serão gerados
options(repr.plot.width=12, repr.plot.height=6)

# Primeiro bloco de características NMF (de 1 a 6)
MapFeatures(SpatialData,
            features = paste0("NMF_", 1:6),              # Seleciona as características NMF_1 a NMF_6
            override_plot_dims = TRUE,                   # Força as dimensões definidas acima
            colors = viridis::magma(n = 11, direction = -1)) &  # Aplica a paleta de cores 'magma' invertida
  theme(plot.title = element_blank())                   # Remove o título do gráfico

# Segundo bloco de características NMF (de 7 a 12)
MapFeatures(SpatialData,
            features = paste0("NMF_", 7:12),             # Seleciona NMF_7 a NMF_12
            override_plot_dims = TRUE,
            colors = viridis::magma(n = 11, direction = -1)) &
  theme(plot.title = element_blank())

# Terceiro bloco de características NMF (de 13 a 18)
MapFeatures(SpatialData,
            features = paste0("NMF_", 13:18),            # Seleciona NMF_13 a NMF_18
            override_plot_dims = TRUE,
            colors = viridis::magma(n = 11, direction = -1)) &
  theme(plot.title = element_blank())

# Quarto bloco de características NMF (de 19 até k)
MapFeatures(SpatialData,
            features = paste0("NMF_", 19:k),             # Seleciona NMF_19 até NMF_k (valor máximo definido)
            override_plot_dims = TRUE,
            colors = viridis::magma(n = 11, direction = -1)) &
  theme(plot.title = element_blank())


Nesta etapa, criamos um gráfico de cargas de características (feature loadings) derivado da redução NMF. Esse tipo de visualização nos permite identificar quais genes contribuem mais fortemente para os primeiros componentes latentes, exibindo os mais relevantes em um gráfico de pontos. Isso facilita a interpretação biológica dos padrões espaciais e a compreensão de como cada gene participa da variabilidade capturada pela NMF.

In [ ]:
# Gera um gráfico de cargas de características (feature loadings) 
# para os dados espaciais usando redução NMF

PlotFeatureLoadings(SpatialData,
                    dims = 1:2,          # Seleciona as duas primeiras dimensões (componentes) para visualizar
                    reduction = "nmf",   # Especifica que a redução utilizada é NMF (Non-negative Matrix Factorization)
                    nfeatures = 30,      # Número de características (genes/variáveis) mais relevantes que serão mostradas
                    mode = "dotplot",    # Define o modo de visualização como um gráfico de pontos (dotplot)
                    fill = "darkmagenta",# Cor de preenchimento para os pontos no gráfico
                    pt_size = 3)         # Tamanho dos pontos na visualização


Nesta etapa, um mapa multicaracterístico de NMFs é gerado sobre os dados espaciais. Essa função permite a visualização simultânea de todos os componentes latentes (NMF_1 a NMF_k) projetados na imagem bruta do tecido. Ajustamos as dimensões dos gráficos e o tamanho dos pontos para obter uma representação clara e uniforme, facilitando a interpretação de como cada padrão de expressão está distribuído espacialmente dentro do contexto do tecido.

In [ ]:
# Ajusta as dimensões dos gráficos que serão gerados
options(repr.plot.width=16, repr.plot.height=9)

# Gera um mapa múltiplo de características NMF nos dados espaciais
MapMultipleFeatures(SpatialData,
            features = paste0("NMF_", 1:k),# Seleciona todas as características NMF de 1 até k
            image_use = "raw",             # Usa a imagem bruta (sem processamento) como fundo de referência
            override_plot_dims = TRUE,     # Força as dimensões definidas acima
            pt_size = 2)                   # Define o tamanho dos pontos na visualização


Nesta visualização, buscamos compreender quais genes são os principais responsáveis ​​por cada componente latente gerado pela redução NMF. Ao visualizar as cargas das características como um mapa de calor, podemos comparar claramente como a contribuição dos genes varia em diferentes dimensões.

In [ ]:
# Gera um mapa de calor das cargas de características (feature loadings)
# utilizando a redução NMF nos dados espaciais

PlotFeatureLoadings(SpatialData,
                    dims = 1:k,        # Seleciona todas as dimensões de 1 até k
                    reduction = "nmf", # Especifica que a redução utilizada é NMF (Non-negative Matrix Factorization)
                    nfeatures = 5,     # Mostra as 5 características mais relevantes por dimensão
                    mode = "heatmap",  # Define o modo de visualização como mapa de calor
                    gradient_colors = viridis::magma(n = 11,  # Aplica a paleta de cores 'magma' da biblioteca viridis
                                                     direction = -1)) # Invertida para destacar melhor os gradientes


In [ ]:
# Gerar UMAP
SpatialData <- RunUMAP(SpatialData, reduction = "nmf", dims = 1:k, verbose = FALSE)

In [ ]:
# Realiza a agrupação baseada em grafos do Seurat
# Encontra os vizinhos mais próximos para cada célula utilizando a fatorização de matriz não negativa (NMF)
SpatialData <- FindNeighbors(object = SpatialData,
                      dims = 1:k, # Usa as primeiras 10 dimensões do método de redução especificado
                      reduction = "nmf", # Indica que foi utilizado NMF como método de redução dimensional
                      verbose = FALSE) # Suprime a saída detalhada

# Define uma sequência de resoluções de 0 até 1.2, com incrementos de 0.2
SpatialData <- FindClusters(SpatialData, resolution = seq(0, 1.2, by = 0.2),
                      verbose = FALSE)


In [ ]:
head(SpatialData@meta.data)

In [ ]:
# Visualiza os resultados de agrupamento a diferentes resoluções usando Clustree
clustree(SpatialData, prefix = "Spatial_snn_res.")

In [ ]:
# Criar um gráfico UMAP agrupado por "ids", sem rótulos de agrupamento (cluster)
DimPlot(SpatialData, reduction = "umap", label = FALSE, group.by = "sample_id",
        pt.size = 2, label.size=13)

In [ ]:
SpatialData2 <- SpatialData
SpatialData2@images[["slice1"]] <- Read10X_Image(dirname(SpatialData2@tools$Staffli@imgs[1]))
SpatialData2@images[["slice2"]] <- Read10X_Image(dirname(SpatialData2@tools$Staffli@imgs[2]))

# Plota apenas a lâmina 3 do paciente G
# Sobrepõe a expressão das características nas coordenadas espaciais
SpatialDimPlot(SpatialData2, group.by = "Spatial_snn_res.0.4") + theme(legend.position = "right")
MapMultipleFeatures(SpatialData2,
                    features = paste0("NMF_", 1:k),
                    image_use = "raw",
                    override_plot_dims = TRUE,
                    pt_size = 2)


Exercício: Salve a imagem em um arquivo.

Visualize diretamente a distribuição de aglomerados no espaço tecidual, sem interferência da imagem histológica de fundo.

In [ ]:
# Plota os clusters sem a imagem histológica (HE) de fundo
SpatialDimPlot(SpatialData2, group.by = "Spatial_snn_res.0.4", image.alpha = 0) + 
  theme(legend.position = "right")   # Visualiza os clusters definidos pela resolução 0.4, sem transparência da imagem

# Adiciona etiquetas aos clusters no mapa espacial
MapLabels(SpatialData, column_name = "Spatial_snn_res.0.4", ncol = 2) & 
  theme(legend.position = "right")   # Coloca a legenda à direita para facilitar a leitura


## Genes Diferencialmente Expressos

Nesta parte da análise, os genes marcadores diferencialmente expressos (DEGs) são identificados entre os diferentes agrupamentos espaciais. A filtragem por significância estatística garante que apenas genes confiáveis ​​sejam retidos, e a contagem final de DEGs por agrupamento permite avaliar quais populações celulares exibem perfis moleculares mais definidos ou enriquecidos.

In [ ]:
devtools::install_github('immunogenomics/presto')

In [ ]:
DefaultAssay(SpatialData) # Verifica o ensaio ativo do objeto Seurat SpatialData
Idents(SpatialData) <- "Spatial_snn_res.0.4" # Define a identidade do objeto de acordo com a resolução de agrupamento (cluster) 0.4

# Comparação de todos os agrupamentos (clusters) entre si
SpatialData.markers <- FindAllMarkers(object = SpatialData, # Identifica genes diferencialmente expressos (DEGs)
                                only.pos = TRUE, # Apenas genes superexpressos nos agrupamentos (clusters)
                                min.pct = 0.10, # O gene deve estar expresso em pelo menos 10% das células
                                logfc.threshold = 0.10) # Limite de log2 fold change

# Filtra os marcadores com p-valor ajustado < 0.05
SpatialData.markers = SpatialData.markers[which(SpatialData.markers$p_val_adj<0.05),] # Filtra os marcadores identificados para manter apenas aqueles com p-valor ajustado

# Conta o número de DEGs por agrupamento (cluster)
table(SpatialData.markers[, "cluster"]) # Quantos DEGs existem por grupo?


In [ ]:
# Seleciona os 10 genes com maior logFC por agrupamento (cluster)
SpatialData.markers %>% group_by(cluster) %>% top_n(n = 10, wt = avg_log2FC) -> top10

# Gera um gráfico de pontos (DotPlot) para os genes selecionados
DotPlot(SpatialData, features = unique(top10$gene),
        group.by = "Spatial_snn_res.0.4", cols = c('#b8d8d8', '#e71d36'),
        dot.scale = 6, col.min = 0) +
        theme(axis.text.x = element_text(face = "bold", color = c("black"),
        size = 8, angle = 90))


Exercício: Salve a imagem em um arquivo.

Esta seção mostra como a expressão varia entre diferentes grupos, ajudando a identificar quais populações celulares são enriquecidas nesses genes, facilitando assim a comparação direta da intensidade e frequência de expressão entre os grupos.

In [ ]:
# Plotar genes-chave em um gráfico de densidade (ridge plot)
RidgePlot(Her2p, assay = "SCT", 
          features = c("IFI27","IFI6"), # Genes de interés
          ncol = 2, group.by = "SCT_snn_res.0.4", 
          cols = color)

In [ ]:
# Visualização UMAP e mapa de calor
# Define o gradiente de cores para o mapa de calor
heatmap.colors <- c("lightgray", "mistyrose", "red", "darkred", "black")
fts <- c("SPAG6","PGM5-AS1") # Lista de genes a serem visualizados

# Gera os gráficos UMAP para cada gene
p.fts <- lapply(fts, function(ftr) {
  FeaturePlot(SpatialData, features = ftr, reduction = "umap", order = TRUE, cols = heatmap.colors, pt.size = 2)
})

# Plota a expressão transformada em coordenadas de Visium
p3 <- MapFeatures(SpatialData, features = fts, ncol = 2, cols = heatmap.colors, pt.size = 2)

# Combina todos os gráficos em uma única figura
cowplot::plot_grid(cowplot::plot_grid(plotlist = p.fts, ncol = 1), p3, ncol = 2, rel_widths = c(1, 1.3))


In [ ]:
# Gráfico de violino para expressão de SPAG6 e PGM5-AS1 agrupado por clusters
# no objeto Seurat 'SpatialData', agrupando pela resolução de clusterização "Spatial_snn_res.0.4".
VlnPlot(SpatialData, features = c("SPAG6", "PGM5-AS1"),
        group.by = "Spatial_snn_res.0.4", # Define a variável de agrupamento (resolução de cluster 0.4)
        pt.size = 0, # Define o tamanho do ponto como 0 para evitar traçar pontos individuais
        ncol = 2) +  # Organiza os gráficos em duas colunas

# Sobrepor uma estatística de resumo (média) como uma barra horizontal
  stat_summary(fun.y = mean, geom = "point", shape = 95,
               size = 15, color = "black") +

# Remover a legenda do gráfico
  NoLegend()


## Visualização 3D


In [ ]:
library(plotly)
library(dplyr)
library(htmlwidgets)

Recupera as coordenadas espaciais de cada ponto ou célula na imagem do tecido e as organiza por amostra. Simultaneamente, obtém informações sobre a imagem associada ao objeto, permitindo que os dados de expressão sejam vinculados à sua posição física no tecido. Isso é fundamental na transcriptômica espacial, que conecta perfis moleculares com a arquitetura tecidual e facilita a interpretação biológica dentro de seu contexto espacial.

In [ ]:
# Obtém as coordenadas espaciais de cada spot/célula na imagem
xy_coords <- GetCoordinates(SpatialData) |>
    # Nota: usam-se pxl_*_in_fullres para a imagem bruta; se as seções estiverem alinhadas,
    # deve-se usar pxl_*_in_fullres_transformed
  dplyr::select(pxl_col_in_fullres, pxl_row_in_fullres, sampleID) |>
  group_by(sampleID) |>
  group_split()   # Separa as coordenadas por amostra

# Extrai informações da imagem associada ao objeto espacial
image_info <- GetImageInfo(SpatialData)


In [ ]:
# Ajusta as coordenadas espaciais de cada amostra
adjusted_coords <- do.call(bind_rows, lapply(seq_along(xy_coords), function(i) {
  xy <- xy_coords[[i]]                        # Extrai as coordenadas da amostra i
  full_width <- image_info[i, ]$full_width    # Obtém a largura total da imagem
  full_height <- image_info[i, ]$full_height  # Obtém a altura total da imagem
  xy <- xy |>
    mutate(x = pxl_col_in_fullres/full_width, # Normaliza a coordenada x em relação à largura
           y = pxl_row_in_fullres/full_height) |> # Normaliza a coordenada y em relação à altura
    # Define um valor z usando o sampleID para separar seções no espaço tridimensional.
    # Esse ajuste permite controlar a distância entre seções alinhadas.
    mutate(z = sampleID*0.2) |>
    dplyr::select(x, y, z)                    # Mantém apenas as coordenadas ajustadas
}))


In [ ]:
# Gráfico de dispersão 3D com plotly
p <- plot_ly(adjusted_coords, x = ~x, y = ~y, z = ~z, type = "scatter3d", mode = "markers", color = SpatialData$Spatial_snn_res.0.4, size = 3)
saveWidget(as_widget(p), "SpatialData3DPlot.html")

## Termos da ontologia genética (GO)

In [ ]:
# Seleciona genes diferencialmente expressos (DEGs) do cluster 6
geneList = SpatialData.markers[which(SpatialData.markers$cluster == 6), "gene"]
length(geneList)  # Mostra a quantidade de genes selecionados

# Converte os símbolos de genes para IDs de Entrez
columns(org.Hs.eg.db)  # Mostra as colunas disponíveis para conversão na base de anotação

symbol <- mapIds(org.Hs.eg.db,
                 keys = geneList,       # Lista de símbolos de genes a converter
                 column = "ENTREZID",   # Converter para IDs de Entrez
                 keytype = "SYMBOL",    # Tipo de entrada: símbolo de gene
                 multiVals = "first")   # Se houver múltiplas correspondências, pega a primeira

symbol = as.vector(symbol)  # Converte para formato vetor

# Remove genes sem ID de Entrez
geneList = symbol
geneList = geneList[which(geneList != "NA")]
length(geneList)  # Mostra a quantidade de genes após o filtro

# Análise de enriquecimento de vias com ReactomePA
x <- enrichPathway(gene = geneList,
                   organism = "human",
                   pvalueCutoff = 0.05,
                   readable = TRUE,
                   pAdjustMethod = "bonferroni")

head(as.data.frame(x))  # Mostra as primeiras linhas dos resultados

# Ordena os resultados por valor ajustado de p (do menor para o maior)
x@result = x@result[order(x@result$p.adjust, decreasing = FALSE),]

# Seleciona as 30 vias mais significativas
x@result = x@result[1:30,]

# Ordena vias por quantidade de genes (do menor para o maior)
x@result = x@result[order(x@result$Count, decreasing = FALSE),]

# Salva os resultados ordenados em um novo data frame
newbar.dt = x@result

# Cria um gráfico de barras com as vias enriquecidas
cluster6_enrichpathway <- ggbarplot(newbar.dt, x = "Description", y = "Count",
          fill = "Count",         # Colore as barras de acordo com o número de genes
          color = "white",        # Cor da borda das barras em branco
          sort.val = "asc",       # Ordena os valores em ordem ascendente
          sort.by.groups = FALSE, # Não ordenar por grupo
          x.text.angle = 90,      # Rotaciona os rótulos do eixo x para melhor visibilidade
          xlab = "Vias",
          ylab = "Número de genes",
          legend.title = "Contagem",
          rotate = TRUE,
          ggtheme = theme_minimal()
) + scale_fill_continuous(low = "#bde0fe", high = "red") +
  theme(text = element_text(size = 20))


## Análise de vias com EnrichR

In [ ]:
# Lista as bases de dados disponíveis no EnrichR
listEnrichrSites()
dbs <- listEnrichrDbs()

# Ordena as bases de dados por nome
dbs %>% dplyr::arrange(libraryName)

# Seleciona bases de dados específicas para a análise de enriquecimento
dbs <- c("Cancer_Cell_Line_Encyclopedia",
         "Elsevier_Pathway_Collection",
         "KEGG_2021_Human",
         "CellMarker_Augmented_2021",
         "Reactome_2016",
         "GO_Biological_Process_2018",
         "GO_Cellular_Component_2018",
         "GO_Molecular_Function_2018",
         "InterPro_Domains_2019")

# EnrichR precisa de símbolos de genes (não IDs de Ensembl)
geneList = SpatialData.markers[which(SpatialData.markers$cluster == 6), "gene"]
length(geneList)  # Mostra a quantidade de genes selecionados

# Realiza a análise de enriquecimento com EnrichR
enriched.Paths <- enrichr(geneList, dbs)

# Mostra resultados das bases de dados selecionadas
enriched.Paths[[1]]  # Cancer_Cell_Line_Encyclopedia
enriched.Paths[[2]]  # Elsevier_Pathway_Collection
enriched.Paths[[3]]  # KEGG_2021_Human
enriched.Paths[[4]]  # CellMarker_Augmented_2021
enriched.Paths[[5]]  # Reactome_2016
enriched.Paths[[6]]  # GO_Biological_Process_2018
enriched.Paths[[7]]  # GO_Cellular_Component_2018
enriched.Paths[[8]]  # GO_Molecular_Function_2018
enriched.Paths[[9]]  # InterPro_Domains_2019

# Seleciona os resultados de CellMarker_Augmented_2021
Rpath = enriched.Paths[[4]]

# Conta a quantidade de genes por cada via
Gene_Count = c()
for(x in 1:dim(Rpath)[1]){
  Gene_Count = c(Gene_Count, str_split(Rpath$Overlap[x], pattern = "/" )[[1]][1])
}

Rpath$Gene_Count = as.numeric(Gene_Count)

# Ordena por valor ajustado de p (do menor para o maior)
Rpath = Rpath[order(Rpath$Adjusted.P.value, decreasing = FALSE),]

# Filtra vias significativas (p ajustado < 0.05)
Rpath = Rpath[Rpath$Adjusted.P.value < 0.05,]

# Seleciona as 20 vias principais
Rpath = Rpath[1:20,]

# Ordena vias por número de genes (do maior para o menor)
Rpath = Rpath[order(Rpath$Gene_Count, decreasing = TRUE),]

# Cria um gráfico de barras das vias enriquecidas
cluster6_enrichR <- ggbarplot(Rpath, x = "Term", y = "Gene_Count",
          fill = "Gene_Count",         # Colore as barras de acordo com o número de genes
          color = "white",             # Cor da borda branca
          sort.val = "asc",            # Ordena os valores em ordem ascendente
          sort.by.groups = FALSE,
          x.text.angle = 90,           # Rotaciona os rótulos para melhor visibilidade
          xlab = "Vias",
          ylab = "Número de genes",
          legend.title = "Contagem",
          rotate = TRUE,
          ggtheme = theme_minimal()
) + scale_fill_continuous(low = "#bde0fe", high = "red") +
  theme(text = element_text(size = 18))

# Mostra o gráfico
cluster6_enrichR


## Deconvolução de Tipos Celulares

A deconvolução é uma técnica computacional usada para inferir e quantificar as proporções de diferentes tipos celulares em uma população celular mista. Como os dados de scRNA-seq frequentemente provêm de tecidos complexos que contêm múltiplos tipos celulares, a deconvolução ajuda a identificar e isolar os perfis de expressão gênica específicos de cada tipo celular dentro da mistura.

Por que a deconvolução é importante?

* Identificação de tipos celulares: A deconvolução permite que os pesquisadores determinem a presença e a abundância de vários tipos celulares em uma amostra de tecido heterogênea.

* Compreensão da composição celular: Ela ajuda a elucidar a composição celular dos tecidos, revelando como diferentes tipos celulares contribuem para processos biológicos e doenças.

* Melhoria da interpretação de dados: Ao separar os sinais de expressão gênica de diferentes tipos celulares, a deconvolução melhora a precisão e a interpretabilidade dos dados de scRNA-seq, levando a insights biológicos mais precisos.

In [ ]:
download.file('https://github.com/integrativebioinformatics/scNotebooks/blob/main/scNotebooks-Resources/scRNASeq_pac2_processed.rds', 'ST_Exercises/scRNASeq_pac2_processed.rds')

In [ ]:
# Carrega o dataset de scRNA-seq
SC.data = readRDS("ST_Exercises/scRNASeq_pac2_processed.rds")
SC.data <-UpdateSeuratObject(SC.data)

# Define identidades das células de acordo com o tipo celular
Idents(SC.data) = "cellType"

# Visualização UMAP dos tipos celulares
DimPlot(object = SC.data, reduction = 'umap', label = TRUE, label.size = 6,
        group.by = "cellType", pt.size = 1.5)

# Identifica DEGs entre todos os agrupamentos (clusters)
SC.markers <- FindAllMarkers(object = SC.data, only.pos = TRUE, min.pct = 0.10, logfc.threshold = 0.10)
# min.pct = 0.10: pelo menos 10% das células devem expressar o gene
# logfc.threshold = 0.10: limite mínimo de log2 fold change para considerar o gene como marcador

SC.markers = SC.markers[which(SC.markers$p_val_adj < 0.05 & SC.markers$avg_log2FC > 0.5),]
# SC.markers$p_val_adj < 0.05: esta condição seleciona marcadores com p-valor ajustado menor que 0.05, ou seja, estatisticamente significativos
# SC.markers$avg_log2FC > 0.5: esta condição seleciona marcadores com log2 fold change médio maior que 0.5

# Conta o número de DEGs por agrupamento (cluster)
table(SC.markers$cluster)

DefaultAssay(SC.data) # retorna o nome do ensaio atualmente definido como padrão para o objeto Seurat


## Fluxo do Pipeline de Deconvolução

In [ ]:
#ti <- Sys.time()
DefaultAssay(SpatialData) <- "Spatial"

# Predição das proporções de tipos celulares
SpatialData <- RunNNLS(object = SpatialData,
                      singlecell_object = SC.data,
                      groups = "cellType")

In [ ]:
# Verifique os tipos de células disponíveis
rownames(SpatialData)

In [ ]:
# Carregar imagens H&E
SpatialData <- SpatialData |>
  LoadImages()

In [ ]:
# Plotar múltiplas características (features)
MapMultipleFeatures(SpatialData,
                    image_use = "raw",
                    pt_size = 2, max_cutoff = 0.95,
                    override_plot_dims = TRUE,
                    features = rownames(SpatialData)) +
  plot_layout(guides = "collect")

MapMultipleFeatures(SpatialData,
                    pt_size = 2, max_cutoff = 0.95,
                    override_plot_dims = TRUE,
                    features = rownames(SpatialData)) +
  plot_layout(guides = "collect")

## Colocalização de Tipos Celulares

A colocalização de tipos celulares refere-se à análise de como diferentes populações celulares estão distribuídas e aparecem juntas em regiões específicas do tecido. Em transcriptômica espacial, essa etapa permite a identificação de padrões de proximidade ou interação entre tipos celulares, revelando potenciais relações funcionais, comunicação celular ou microambientes relevantes para processos biológicos e doenças.

In [ ]:
library(pheatmap)

# Extrai a matriz de expressão de SpatialData e calcula a correlação entre genes/tipos celulares
cor_matrix <- FetchData(SpatialData, rownames(SpatialData)) |>
  mutate_all(~ if_else(.x<0.1, 0, .x)) |>  # Filtra valores muito baixos, definindo-os como 0
  cor()                                    # Calcula a matriz de correlação

diag(cor_matrix) <- NA                     # Remove a diagonal (autocorrelações) para maior clareza
max_val <- max(cor_matrix, na.rm = T)      # Obtém o valor máximo de correlação

# Define a paleta de cores para o heatmap (vermelho-amarelo-azul invertida, com branco no centro)
cols <- RColorBrewer::brewer.pal(7, "RdYlBu") |> rev(); cols[4] <- "white"

# Ajusta as dimensões da figura
options(repr.plot.width=6, repr.plot.height=6)

# Gera o heatmap de correlações entre tipos celulares dentro dos spots
pheatmap::pheatmap(cor_matrix,
                   breaks = seq(-max_val, max_val, length.out = 100), # Intervalo de valores de correlação
                   color=colorRampPalette(cols)(100),                 # Gradiente de cores
                   cellwidth = 14, cellheight = 14,                   # Tamanho das células
                   treeheight_col = 10, treeheight_row = 10,          # Altura dos dendrogramas
                   main = "Correlação entre tipos celulares\nnos spots") # Título do gráfico


In [ ]:
# Aplica a non-negative matrix factorization (NMF) sobre os dados espaciais
nmf_data <- FetchData(SpatialData, rownames(SpatialData)) |>   # Extrai a matriz de expressão do objeto espacial
  RcppML::nmf(k = 10, verbose = T)                             # Executa NMF com k = 10 componentes latentes, mostrando mensagens no console


In [ ]:
# Converte a matriz H do resultado NMF em um data.frame
nmf_data_h <- nmf_data@h |> as.data.frame()

# Atribui nomes de linha aos fatores (Factor_1 a Factor_10)
rownames(nmf_data_h) <- paste0("Factor_", 1:10)

# Atribui nomes de coluna correspondentes às células/spots do objeto espacial
colnames(nmf_data_h) <- rownames(SpatialData)

# Normaliza os valores de cada coluna dividindo pelo valor máximo (entre 0 e 1)
nmf_data_h <- nmf_data_h |>
  mutate_at(colnames(nmf_data_h),
            ~(scale(., center = FALSE, scale = max(., na.rm = TRUE)/1)))

# Cria uma coluna Factor com os nomes das linhas, mantendo a ordem dos fatores
nmf_data_h$Factor <- rownames(nmf_data_h) |>
  factor(levels = paste0("Factor_", 1:10))

# Reorganiza os dados em formato longo (long format):
# cada linha representa o peso (Weight) de um fator em uma célula específica
nmf_data_h_df <- nmf_data_h |>
  tidyr::pivot_longer(cols = all_of(rownames(SpatialData)),
                      names_to = "Cell",
                      values_to = "Weight")


In [ ]:
# Bubble chart
ggplot(nmf_data_h_df, aes(x=Factor, y=Cell, size=Weight, color=Weight)) +
  geom_point() +
  labs(title="Cell type contribution", x="Factor", y = "Cell type",
       color = "", size = "Scaled weight") +
  scale_color_viridis_c(direction = -1, option = "magma") +
  theme_bw() +
  theme(axis.text.x = element_text(angle=45, hjust=1),
        panel.grid = element_blank())